In [1]:
"""
Comprehensive Synthetic Data Generator for Nepal Maternal Mental Health
Based on Nepal DHS 2022, Census 2021, WHO Reports, and Research Literature
Generates 100,000+ realistic records with complex inter-feature dependencies
"""

import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

class NepalMaternalHealthDataGenerator:
    def __init__(self, n_samples=100000, random_seed=42):
        """
        Initialize data generator with Nepal-specific distributions
        Based on comprehensive research from DHS 2022, Census 2021, WHO reports
        """
        self.n_samples = n_samples
        np.random.seed(random_seed)
        
        # Province distributions (Census 2021)
        self.provinces = {
            'Koshi': 0.169,
            'Madhesh': 0.220,
            'Bagmati': 0.184,
            'Gandaki': 0.095,
            'Lumbini': 0.177,
            'Karnali': 0.062,
            'Sudurpashchim': 0.093
        }
        
        # Ethnicity distributions (Census 2021, adjusted for maternal population)
        self.ethnicities = {
            'Brahmin/Chhetri': 0.278,  # Combined Hill Brahmin (11.29%) + Chhetri (16.45%)
            'Janajati': 0.368,  # Various indigenous groups
            'Madheshi': 0.185,  # Terai groups
            'Dalit': 0.130,     # Marginalized groups
            'Muslim': 0.039     # Religious minority
        }
        
        # Education levels (DHS 2022)
        self.education_levels = {
            'No education': 0.15,
            'Primary': 0.22,
            'Secondary': 0.43,
            'Higher': 0.20
        }
        
    def generate_demographics(self):
        """Generate demographic features with realistic distributions"""
        print("Generating demographic features...")
        
        # Age: Normal distribution centered at 26, SD 5.2 (DHS 2022)
        age = np.clip(np.random.normal(26, 5.2, self.n_samples), 15, 49).astype(int)
        
        # Residence: 40% urban, 60% rural (Census 2021)
        residence = np.random.choice(['Urban', 'Rural'], 
                                    self.n_samples, 
                                    p=[0.40, 0.60])
        
        # Province allocation
        province = np.random.choice(
            list(self.provinces.keys()),
            self.n_samples,
            p=list(self.provinces.values())
        )
        
        # Ethnicity distribution
        ethnicity = np.random.choice(
            list(self.ethnicities.keys()),
            self.n_samples,
            p=list(self.ethnicities.values())
        )
        
        # Education correlated with age, residence, and ethnicity
        education = self._generate_education(age, residence, ethnicity)
        
        # Occupation based on education and residence
        occupation = self._generate_occupation(education, residence)
        
        # Family type: Nuclear 45%, Joint 50%, Extended 5%
        family_type = np.random.choice(
            ['Nuclear', 'Joint', 'Extended'],
            self.n_samples,
            p=[0.45, 0.50, 0.05]
        )
        
        # Caste discrimination exposure (higher for Dalit and marginalized groups)
        discrimination_prob = np.where(
            ethnicity == 'Dalit', 0.45,
            np.where(ethnicity == 'Muslim', 0.35, 0.12)
        )
        caste_discrimination = np.random.binomial(1, discrimination_prob).astype(bool)
        
        return pd.DataFrame({
            'age': age,
            'residence': residence,
            'province': province,
            'ethnicity': ethnicity,
            'education_level': education,
            'occupation': occupation,
            'family_type': family_type,
            'caste_discrimination_exposure': caste_discrimination
        })
    
    def _generate_education(self, age, residence, ethnicity):
        """Generate education with correlations"""
        education = []
        for i in range(len(age)):
            # Base probabilities
            probs = [0.15, 0.22, 0.43, 0.20]
            
            # Adjust for age (younger more educated)
            if age[i] < 25:
                probs = [0.08, 0.18, 0.48, 0.26]
            elif age[i] > 35:
                probs = [0.22, 0.28, 0.38, 0.12]
            
            # Adjust for urban (more educated)
            if residence[i] == 'Urban':
                probs = [p * 0.6 if idx == 0 else p * 1.15 for idx, p in enumerate(probs)]
            
            # Adjust for ethnicity
            if ethnicity[i] in ['Dalit', 'Muslim']:
                probs = [p * 1.4 if idx == 0 else p * 0.85 for idx, p in enumerate(probs)]
            
            # Normalize
            probs = np.array(probs) / sum(probs)
            education.append(np.random.choice(
                list(self.education_levels.keys()), p=probs
            ))
        
        return education
    
    def _generate_occupation(self, education, residence):
        """Generate occupation based on education and residence"""
        occupation = []
        for i in range(len(education)):
            if education[i] == 'No education':
                occ = np.random.choice(
                    ['Unemployed', 'Agriculture', 'Unskilled labor'],
                    p=[0.40, 0.50, 0.10]
                )
            elif education[i] in ['Primary', 'Secondary']:
                if residence[i] == 'Rural':
                    occ = np.random.choice(
                        ['Agriculture', 'Unemployed', 'Skilled labor', 'Business'],
                        p=[0.45, 0.25, 0.20, 0.10]
                    )
                else:
                    occ = np.random.choice(
                        ['Employed', 'Business', 'Unemployed', 'Skilled labor'],
                        p=[0.35, 0.25, 0.25, 0.15]
                    )
            else:  # Higher education
                occ = np.random.choice(
                    ['Employed', 'Business', 'Professional', 'Unemployed'],
                    p=[0.50, 0.25, 0.15, 0.10]
                )
            occupation.append(occ)
        
        return occupation
    
    def generate_socioeconomic(self, demographics):
        """Generate socioeconomic features with dependencies"""
        print("Generating socioeconomic features...")
        
        # Wealth index based on residence, province, ethnicity
        wealth_index = self._generate_wealth_index(demographics)
        
        # Monthly income (NPR) - log-normal, correlated with wealth
        income = self._generate_income(wealth_index, demographics)
        
        # Food security based on wealth and income
        food_security = self._generate_food_security(wealth_index, income)
        
        # Housing quality (1-10) based on wealth and residence
        housing_quality = self._generate_housing_quality(wealth_index, demographics['residence'])
        
        # Financial stress (0-10) inversely correlated with income
        financial_stress = self._generate_financial_stress(income, wealth_index)
        
        return pd.DataFrame({
            'wealth_index': wealth_index,
            'monthly_household_income_npr': income,
            'food_security': food_security,
            'housing_quality': housing_quality,
            'financial_stress_level': financial_stress
        })
    
    def _generate_wealth_index(self, demographics):
        """Generate wealth index with regional variations"""
        wealth = []
        for i in range(len(demographics)):
            # Base probabilities
            probs = [0.20, 0.20, 0.20, 0.20, 0.20]
            
            # Urban areas are wealthier
            if demographics.iloc[i]['residence'] == 'Urban':
                probs = [0.08, 0.12, 0.20, 0.28, 0.32]
            else:
                probs = [0.28, 0.26, 0.22, 0.16, 0.08]
            
            # Province variations (Bagmati and Gandaki wealthier, Karnali poorer)
            if demographics.iloc[i]['province'] in ['Bagmati', 'Gandaki']:
                probs = [p * 0.7 if idx < 2 else p * 1.2 for idx, p in enumerate(probs)]
            elif demographics.iloc[i]['province'] in ['Karnali', 'Madhesh']:
                probs = [p * 1.4 if idx < 2 else p * 0.7 for idx, p in enumerate(probs)]
            
            # Ethnicity effects
            if demographics.iloc[i]['ethnicity'] in ['Dalit', 'Muslim']:
                probs = [p * 1.3 if idx < 2 else p * 0.8 for idx, p in enumerate(probs)]
            elif demographics.iloc[i]['ethnicity'] == 'Brahmin/Chhetri':
                probs = [p * 0.8 if idx < 2 else p * 1.15 for idx, p in enumerate(probs)]
            
            probs = np.array(probs) / sum(probs)
            wealth.append(np.random.choice(
                ['Poorest', 'Poorer', 'Middle', 'Richer', 'Richest'], p=probs
            ))
        
        return wealth
    
    def _generate_income(self, wealth_index, demographics):
        """Generate monthly household income (NPR)"""
        income_map = {
            'Poorest': (10000, 0.35),
            'Poorer': (18000, 0.40),
            'Middle': (35000, 0.45),
            'Richer': (60000, 0.50),
            'Richest': (120000, 0.60)
        }
        
        income = []
        for i, w in enumerate(wealth_index):
            base, std_factor = income_map[w]
            # Log-normal distribution
            inc = np.random.lognormal(np.log(base), std_factor)
            
            # Urban premium
            if demographics.iloc[i]['residence'] == 'Urban':
                inc *= 1.35
            
            # Education bonus
            edu = demographics.iloc[i]['education_level']
            if edu == 'Higher':
                inc *= 1.4
            elif edu == 'Secondary':
                inc *= 1.15
            
            income.append(np.clip(inc, 8000, 250000))
        
        return np.array(income).astype(int)
    
    def _generate_food_security(self, wealth_index, income):
        """Generate food security status"""
        food_sec = []
        for w, inc in zip(wealth_index, income):
            if w in ['Poorest', 'Poorer'] or inc < 20000:
                fs = np.random.choice(
                    ['Secure', 'Mildly insecure', 'Moderately insecure', 'Severely insecure'],
                    p=[0.15, 0.25, 0.35, 0.25]
                )
            elif w == 'Middle':
                fs = np.random.choice(
                    ['Secure', 'Mildly insecure', 'Moderately insecure', 'Severely insecure'],
                    p=[0.45, 0.30, 0.20, 0.05]
                )
            else:
                fs = np.random.choice(
                    ['Secure', 'Mildly insecure', 'Moderately insecure', 'Severely insecure'],
                    p=[0.75, 0.18, 0.05, 0.02]
                )
            food_sec.append(fs)
        
        return food_sec
    
    def _generate_housing_quality(self, wealth_index, residence):
        """Generate housing quality score (1-10)"""
        quality_map = {
            'Poorest': (3, 1.2),
            'Poorer': (4, 1.3),
            'Middle': (6, 1.4),
            'Richer': (7.5, 1.2),
            'Richest': (9, 0.8)
        }
        
        quality = []
        for w, res in zip(wealth_index, residence):
            mean, std = quality_map[w]
            if res == 'Urban':
                mean += 0.8
            q = np.random.normal(mean, std)
            quality.append(np.clip(q, 1, 10))
        
        return np.array(quality).round(1)
    
    def _generate_financial_stress(self, income, wealth_index):
        """Generate financial stress level (0-10)"""
        stress = []
        wealth_map = {'Poorest': 7.5, 'Poorer': 6.5, 'Middle': 5, 'Richer': 3.5, 'Richest': 2}
        
        for inc, w in zip(income, wealth_index):
            base = wealth_map[w]
            # Income adjustment
            if inc < 15000:
                base += 1.5
            elif inc > 80000:
                base -= 1.5
            
            s = np.random.normal(base, 1.5)
            stress.append(np.clip(s, 0, 10))
        
        return np.array(stress).round(1)
    
    def generate_obstetric(self, demographics):
        """Generate obstetric and clinical features"""
        print("Generating obstetric features...")
        
        # Parity (number of previous births) - Poisson(2.3)
        parity = np.random.poisson(2.3, self.n_samples)
        parity = np.clip(parity, 0, 8)
        
        # ANC visits (based on DHS 2022: 80.2% had 4+)
        anc_visits = self._generate_anc_visits(demographics, parity)
        
        # Delivery type (DHS 2022: 18.2% CS)
        delivery_type = self._generate_delivery_type(demographics, parity, anc_visits)
        
        # Birth complications
        birth_complications = self._generate_complications(delivery_type, demographics, parity)
        
        # Baby health status
        baby_health = self._generate_baby_health(birth_complications, delivery_type)
        
        # Gestational age (weeks)
        gestational_age = np.clip(np.random.normal(39, 2, self.n_samples), 28, 43).astype(int)
        
        # Maternal age at first birth
        age_first_birth = demographics['age'] - (parity * 2.5 + np.random.normal(0, 1.5, self.n_samples))
        age_first_birth = np.clip(age_first_birth, 14, 40).astype(int)
        
        # Previous pregnancy loss (20%)
        prev_loss = np.random.binomial(1, 0.20, self.n_samples).astype(bool)
        
        return pd.DataFrame({
            'parity': parity,
            'anc_visits': anc_visits,
            'delivery_type': delivery_type,
            'birth_complications': birth_complications,
            'baby_health_status': baby_health,
            'gestational_age': gestational_age,
            'maternal_age_at_first_birth': age_first_birth,
            'previous_pregnancy_loss': prev_loss
        })
    
    def _generate_anc_visits(self, demographics, parity):
        """Generate ANC visits (DHS 2022: 80.2% had 4+)"""
        anc = []
        for i in range(len(demographics)):
            # Base: 80% get 4+ visits
            if demographics.iloc[i]['education_level'] in ['Secondary', 'Higher']:
                prob_4plus = 0.88
            elif demographics.iloc[i]['residence'] == 'Urban':
                prob_4plus = 0.85
            else:
                prob_4plus = 0.72
            
            if np.random.random() < prob_4plus:
                visits = np.random.choice([4, 5, 6, 7, 8, 9, 10], p=[0.25, 0.22, 0.18, 0.15, 0.10, 0.06, 0.04])
            else:
                visits = np.random.choice([0, 1, 2, 3], p=[0.05, 0.12, 0.28, 0.55])
            
            anc.append(visits)
        
        return anc
    
    def _generate_delivery_type(self, demographics, parity, anc_visits):
        """Generate delivery type (CS 18.2% in 2022)"""
        delivery = []
        for i in range(len(demographics)):
            # Base CS rate 18.2%
            cs_prob = 0.182
            
            # Urban: higher CS
            if demographics.iloc[i]['residence'] == 'Urban':
                cs_prob = 0.28
            else:
                cs_prob = 0.12
            
            # Wealthy: higher CS
            # Age factors
            age = demographics.iloc[i]['age']
            if age > 35 or age < 20:
                cs_prob *= 1.4
            
            # Previous CS (repeat CS)
            if parity[i] > 0:
                cs_prob *= 1.3
            
            # Poor ANC
            if anc_visits[i] < 4:
                cs_prob *= 0.7
            
            cs_prob = min(cs_prob, 0.55)
            
            if np.random.random() < cs_prob:
                dtype = 'Cesarean'
            else:
                dtype = np.random.choice(['Normal', 'Assisted'], p=[0.92, 0.08])
            
            delivery.append(dtype)
        
        return delivery
    
    def _generate_complications(self, delivery_type, demographics, parity):
        """Generate birth complications"""
        complications = []
        for i, dt in enumerate(delivery_type):
            # Base complication rate 25%
            comp_prob = 0.25
            
            if dt == 'Cesarean':
                comp_prob = 0.35
            elif dt == 'Assisted':
                comp_prob = 0.32
            
            # Age risks
            age = demographics.iloc[i]['age']
            if age > 35 or age < 18:
                comp_prob *= 1.6
            
            # Primipara risk
            if parity[i] == 0:
                comp_prob *= 1.3
            
            complications.append(np.random.random() < comp_prob)
        
        return complications
    
    def _generate_baby_health(self, complications, delivery_type):
        """Generate baby health status"""
        health = []
        for comp, dt in zip(complications, delivery_type):
            if comp:
                status = np.random.choice(
                    ['Healthy', 'Minor issues', 'Major issues'],
                    p=[0.55, 0.30, 0.15]
                )
            else:
                status = np.random.choice(
                    ['Healthy', 'Minor issues', 'Major issues'],
                    p=[0.88, 0.10, 0.02]
                )
            health.append(status)
        
        return health
    
    def generate_psychosocial(self, demographics, obstetric):
        """Generate psychosocial features"""
        print("Generating psychosocial features...")
        
        # Partner support (0-10)
        partner_support = self._generate_partner_support(demographics)
        
        # Family support (0-10)
        family_support = self._generate_family_support(demographics, partner_support)
        
        # Domestic violence (DHS 2022: 27.2% any IPV)
        dv_exposure = self._generate_domestic_violence(demographics, partner_support)
        
        # Social isolation (0-10)
        social_isolation = self._generate_social_isolation(demographics, dv_exposure)
        
        # Sleep quality (0-10)
        sleep_quality = self._generate_sleep_quality(obstetric, demographics)
        
        # Stressful life events (0-5)
        stressful_events = self._generate_stressful_events(demographics)
        
        # Mental health awareness
        mh_awareness = self._generate_mh_awareness(demographics)
        
        return pd.DataFrame({
            'partner_support_score': partner_support,
            'family_support_score': family_support,
            'domestic_violence_exposure': dv_exposure,
            'social_isolation_score': social_isolation,
            'sleep_quality': sleep_quality,
            'stressful_life_events': stressful_events,
            'mental_health_awareness': mh_awareness
        })
    
    def _generate_partner_support(self, demographics):
        """Generate partner support score"""
        support = []
        for i in range(len(demographics)):
            # Base mean 6.5
            mean = 6.5
            
            # Joint family: slightly lower partner support
            if demographics.iloc[i]['family_type'] == 'Joint':
                mean -= 0.8
            
            # Education correlation
            if demographics.iloc[i]['education_level'] == 'Higher':
                mean += 0.7
            
            s = np.random.normal(mean, 2.2)
            support.append(np.clip(s, 0, 10))
        
        return np.array(support).round(1)
    
    def _generate_family_support(self, demographics, partner_support):
        """Generate family support score"""
        support = []
        for i in range(len(demographics)):
            # Higher in joint families
            if demographics.iloc[i]['family_type'] == 'Joint':
                mean = 7.5
            elif demographics.iloc[i]['family_type'] == 'Extended':
                mean = 8.0
            else:
                mean = 5.5
            
            # Some correlation with partner support
            if partner_support[i] > 7:
                mean += 0.5
            
            s = np.random.normal(mean, 2.0)
            support.append(np.clip(s, 0, 10))
        
        return np.array(support).round(1)
    
    def _generate_domestic_violence(self, demographics, partner_support):
        """Generate DV exposure (DHS 2022: 27.2% any form)"""
        dv = []
        for i in range(len(demographics)):
            # Base: 27.2% experience IPV
            dv_prob = 0.272
            
            # Low partner support increases risk
            if partner_support[i] < 4:
                dv_prob = 0.55
            elif partner_support[i] < 6:
                dv_prob = 0.38
            elif partner_support[i] > 8:
                dv_prob = 0.12
            
            # Ethnicity risks (Dalit and marginalized groups)
            if demographics.iloc[i]['ethnicity'] in ['Dalit', 'Madheshi']:
                dv_prob *= 1.25
            
            # Low education
            if demographics.iloc[i]['education_level'] == 'No education':
                dv_prob *= 1.35
            
            dv_prob = min(dv_prob, 0.70)
            
            if np.random.random() < dv_prob:
                exp = np.random.choice(
                    ['Never', 'Rarely', 'Sometimes', 'Often'],
                    p=[0.15, 0.35, 0.35, 0.15]
                )
            else:
                exp = 'Never'
            
            dv.append(exp)
        
        return dv
    
    def _generate_social_isolation(self, demographics, dv_exposure):
        """Generate social isolation score"""
        isolation = []
        for i in range(len(demographics)):
            # Base mean 4.5
            mean = 4.5
            
            # Nuclear families: higher isolation
            if demographics.iloc[i]['family_type'] == 'Nuclear':
                mean += 1.5
            
            # Urban: higher isolation
            if demographics.iloc[i]['residence'] == 'Urban':
                mean += 1.0
            
            # DV increases isolation
            if dv_exposure[i] in ['Sometimes', 'Often']:
                mean += 2.0
            
            iso = np.random.normal(mean, 2.0)
            isolation.append(np.clip(iso, 0, 10))
        
        return np.array(isolation).round(1)
    
    def _generate_sleep_quality(self, obstetric, demographics):
        """Generate sleep quality score"""
        quality = []
        for i in range(len(obstetric)):
            # Base mean 6.0
            mean = 6.0
            
            # Baby health affects sleep
            if obstetric.iloc[i]['baby_health_status'] == 'Major issues':
                mean -= 3.0
            elif obstetric.iloc[i]['baby_health_status'] == 'Minor issues':
                mean -= 1.5
            
            # Complications affect sleep
            if obstetric.iloc[i]['birth_complications']:
                mean -= 1.2
            
            # First-time mothers
            if obstetric.iloc[i]['parity'] == 0:
                mean -= 0.8
            
            q = np.random.normal(mean, 1.8)
            quality.append(np.clip(q, 0, 10))
        
        return np.array(quality).round(1)
    
    def _generate_stressful_events(self, demographics):
        """Generate count of stressful life events"""
        events = []
        for i in range(len(demographics)):
            # Base: Poisson(1.2)
            base_rate = 1.2
            
            # Poor socioeconomic status
            if demographics.iloc[i].get('wealth_index') in ['Poorest', 'Poorer']:
                base_rate = 1.8
            
            count = np.random.poisson(base_rate)
            events.append(min(count, 5))
        
        return events
    
    def _generate_mh_awareness(self, demographics):
        """Generate mental health awareness"""
        awareness = []
        for i in range(len(demographics)):
            edu = demographics.iloc[i]['education_level']
            
            if edu == 'Higher':
                aw = np.random.choice(['Low', 'Medium', 'High'], p=[0.10, 0.35, 0.55])
            elif edu == 'Secondary':
                aw = np.random.choice(['Low', 'Medium', 'High'], p=[0.25, 0.50, 0.25])
            else:
                aw = np.random.choice(['Low', 'Medium', 'High'], p=[0.60, 0.30, 0.10])
            
            awareness.append(aw)
        
        return awareness
    
    def generate_healthcare_access(self, demographics):
        """Generate healthcare access features"""
        print("Generating healthcare access features...")
        
        # Distance to health facility (km)
        distance = []
        for i in range(len(demographics)):
            if demographics.iloc[i]['residence'] == 'Urban':
                # Urban: log-normal(1.5, 0.8)
                dist = np.random.lognormal(0.4, 0.7)
            else:
                # Rural: log-normal(8, 1.2)
                dist = np.random.lognormal(2.0, 1.0)
            
            # Province effects
            if demographics.iloc[i]['province'] in ['Karnali', 'Sudurpashchim']:
                dist *= 1.6
            
            distance.append(np.clip(dist, 0.5, 80))
        
        distance = np.array(distance).round(1)
        
        # Health insurance (30% coverage)
        insurance = []
        for i in range(len(demographics)):
            prob = 0.30
            if demographics.iloc[i]['education_level'] in ['Secondary', 'Higher']:
                prob = 0.42
            if demographics.iloc[i]['residence'] == 'Urban':
                prob *= 1.25
            
            insurance.append(np.random.random() < min(prob, 0.65))
        
        # Previous mental health consultation (5%)
        prev_mh = np.random.binomial(1, 0.05, self.n_samples).astype(bool)
        
        return pd.DataFrame({
            'distance_to_health_facility_km': distance,
            'health_insurance': insurance,
            'previous_mental_health_consultation': prev_mh
        })
    
    def calculate_ppd_risk(self, demographics, socioeconomic, obstetric, psychosocial, healthcare):
        """
        Calculate PPD risk based on research literature
        Target: 20-30% at moderate-to-very-high risk (matching Nepal prevalence)
        """
        print("Calculating PPD risk scores...")
        
        # Combine all features
        df = pd.concat([demographics, socioeconomic, obstetric, psychosocial, healthcare], axis=1)
        
        # Initialize base risk (15%)
        base_risk = np.full(len(df), 0.15)
        risk_scores = base_risk.copy()
        
        # Apply multiplicative risk factors based on literature
        for i in range(len(df)):
            risk = base_risk[i]
            
            # AGE FACTORS
            age = df.iloc[i]['age']
            if age < 18:
                risk *= 1.6  # Teenage mothers
            elif age > 40:
                risk *= 1.4  # Advanced maternal age
            elif age < 20:
                risk *= 1.3
            
            # PARITY FACTORS
            if df.iloc[i]['parity'] == 0:
                risk *= 1.35  # First-time mothers (OR: 1.2-1.5)
            elif df.iloc[i]['parity'] > 4:
                risk *= 1.25  # Grand multiparity
            
            # OBSTETRIC COMPLICATIONS
            if df.iloc[i]['birth_complications']:
                risk *= 1.9  # Complications (OR: 1.8-2.5)
            
            if df.iloc[i]['baby_health_status'] == 'Major issues':
                risk *= 2.4  # Unhealthy baby (OR: 2.0-2.8)
            elif df.iloc[i]['baby_health_status'] == 'Minor issues':
                risk *= 1.5
            
            if df.iloc[i]['delivery_type'] == 'Cesarean':
                risk *= 1.3  # CS delivery
            
            if df.iloc[i]['previous_pregnancy_loss']:
                risk *= 1.4  # Previous loss (OR: 1.3)
            
            # PSYCHOSOCIAL FACTORS (Strongest predictors)
            partner_support = df.iloc[i]['partner_support_score']
            if partner_support < 3:
                risk *= 2.5  # Very poor support (OR: 1.9-3.2)
            elif partner_support < 5:
                risk *= 1.9
            elif partner_support < 6:
                risk *= 1.4
            
            family_support = df.iloc[i]['family_support_score']
            if family_support < 4:
                risk *= 1.7
            elif family_support < 6:
                risk *= 1.3
            
            # DOMESTIC VIOLENCE (OR: 2.5-4.0)
            dv = df.iloc[i]['domestic_violence_exposure']
            if dv == 'Often':
                risk *= 3.2
            elif dv == 'Sometimes':
                risk *= 2.5
            elif dv == 'Rarely':
                risk *= 1.6
            
            # SOCIAL ISOLATION (OR: 1.6-2.3)
            if df.iloc[i]['social_isolation_score'] > 7:
                risk *= 2.0
            elif df.iloc[i]['social_isolation_score'] > 5:
                risk *= 1.5
            
            # SLEEP QUALITY (OR: 1.5-2.0)
            if df.iloc[i]['sleep_quality'] < 3:
                risk *= 1.9
            elif df.iloc[i]['sleep_quality'] < 5:
                risk *= 1.5
            
            # STRESSFUL LIFE EVENTS (OR: 1.8)
            if df.iloc[i]['stressful_life_events'] >= 3:
                risk *= 1.8
            elif df.iloc[i]['stressful_life_events'] >= 2:
                risk *= 1.4
            
            # SOCIOECONOMIC FACTORS
            if df.iloc[i]['wealth_index'] in ['Poorest', 'Poorer']:
                risk *= 1.5  # Poverty (OR: 1.4-1.8)
            
            if df.iloc[i]['food_security'] == 'Severely insecure':
                risk *= 1.6
            elif df.iloc[i]['food_security'] in ['Moderately insecure']:
                risk *= 1.3
            
            if df.iloc[i]['financial_stress_level'] > 7:
                risk *= 1.4
            
            # EDUCATION & AWARENESS
            if df.iloc[i]['education_level'] == 'No education':
                risk *= 1.25  # Low education
            
            # DISCRIMINATION & STIGMA
            if df.iloc[i]['caste_discrimination_exposure']:
                risk *= 1.3
            
            # HEALTHCARE ACCESS
            if df.iloc[i]['distance_to_health_facility_km'] > 20:
                risk *= 1.2  # Poor access
            
            if df.iloc[i]['anc_visits'] < 4:
                risk *= 1.3  # Inadequate prenatal care
            
            # PROTECTIVE FACTORS (reduce risk)
            if df.iloc[i]['mental_health_awareness'] == 'High':
                risk *= 0.85
            
            if df.iloc[i]['health_insurance']:
                risk *= 0.90
            
            if df.iloc[i]['previous_mental_health_consultation']:
                risk *= 0.88
            
            # Cap maximum risk
            risk = min(risk, 0.85)
            risk_scores[i] = risk
        
        # Convert risk probability to EPDS-equivalent score (0-30)
        # Using logistic transformation
        epds_scores = -10 * np.log((1 - risk_scores) / risk_scores)
        epds_scores = np.clip(epds_scores, 0, 30)
        
        # Add individual variation (Gaussian noise)
        epds_scores += np.random.normal(0, 2.5, len(epds_scores))
        epds_scores = np.clip(epds_scores, 0, 30)
        
        # Classify into risk levels
        risk_levels = []
        for score in epds_scores:
            if score < 10:
                level = 'Low Risk'
            elif score < 13:
                level = 'Moderate Risk'
            elif score < 16:
                level = 'High Risk'
            else:
                level = 'Very High Risk'
            risk_levels.append(level)
        
        return pd.DataFrame({
            'epds_equivalent_score': epds_scores.round(1),
            'ppd_risk_level': risk_levels,
            'risk_probability': risk_scores.round(4)
        })
    
    def generate_complete_dataset(self):
        """Generate complete synthetic dataset"""
        print(f"\n{'='*70}")
        print(f"NEPAL MATERNAL MENTAL HEALTH SYNTHETIC DATA GENERATOR")
        print(f"Target sample size: {self.n_samples:,}")
        print(f"{'='*70}\n")
        
        # Generate all feature groups
        demographics = self.generate_demographics()
        socioeconomic = self.generate_socioeconomic(demographics)
        obstetric = self.generate_obstetric(demographics)
        
        # Merge for psychosocial generation (needs wealth_index)
        temp_df = pd.concat([demographics, socioeconomic], axis=1)
        psychosocial = self.generate_psychosocial(temp_df, obstetric)
        
        healthcare = self.generate_healthcare_access(demographics)
        
        # Calculate PPD risk
        ppd_risk = self.calculate_ppd_risk(
            demographics, socioeconomic, obstetric, psychosocial, healthcare
        )
        
        # Combine all features
        complete_data = pd.concat([
            demographics, 
            socioeconomic, 
            obstetric, 
            psychosocial, 
            healthcare,
            ppd_risk
        ], axis=1)
        
        # Add metadata
        complete_data['record_id'] = ['NPL_' + str(i).zfill(6) for i in range(1, len(complete_data) + 1)]
        
        # Reorder columns
        cols = ['record_id'] + [col for col in complete_data.columns if col != 'record_id']
        complete_data = complete_data[cols]
        
        print(f"\n{'='*70}")
        print("DATA GENERATION COMPLETE!")
        print(f"{'='*70}")
        
        return complete_data
    
    def validate_dataset(self, data):
        """Validate synthetic data against Nepal statistics"""
        print(f"\n{'='*70}")
        print("DATASET VALIDATION REPORT")
        print(f"{'='*70}\n")
        
        print(f"Total Records: {len(data):,}\n")
        
        # Demographic validation
        print("=== DEMOGRAPHIC DISTRIBUTIONS ===")
        print(f"Mean Age: {data['age'].mean():.1f} (Target: ~26)")
        print(f"Urban: {(data['residence'] == 'Urban').mean()*100:.1f}% (Target: 40%)")
        print(f"Rural: {(data['residence'] == 'Rural').mean()*100:.1f}% (Target: 60%)\n")
        
        # Education
        print("Education Distribution:")
        print(data['education_level'].value_counts(normalize=True).mul(100).round(1))
        print()
        
        # Obstetric validation
        print("=== OBSTETRIC STATISTICS ===")
        print(f"Mean Parity: {data['parity'].mean():.2f} (Target: ~2.3)")
        print(f"4+ ANC Visits: {(data['anc_visits'] >= 4).mean()*100:.1f}% (Target: 80.2%)")
        print(f"Cesarean Rate: {(data['delivery_type'] == 'Cesarean').mean()*100:.1f}% (Target: 18.2%)")
        print(f"Birth Complications: {data['birth_complications'].mean()*100:.1f}% (Target: ~25%)\n")
        
        # Psychosocial validation
        print("=== PSYCHOSOCIAL FACTORS ===")
        print(f"Any Domestic Violence: {(data['domestic_violence_exposure'] != 'Never').mean()*100:.1f}% (Target: 27.2%)")
        print(f"Mean Partner Support: {data['partner_support_score'].mean():.1f}/10")
        print(f"Mean Social Isolation: {data['social_isolation_score'].mean():.1f}/10\n")
        
        # PPD Risk Distribution
        print("=== PPD RISK DISTRIBUTION ===")
        risk_dist = data['ppd_risk_level'].value_counts(normalize=True).mul(100).round(1)
        print(risk_dist)
        
        at_risk = (data['ppd_risk_level'].isin(['Moderate Risk', 'High Risk', 'Very High Risk'])).mean() * 100
        print(f"\nTotal At Risk (Moderate+): {at_risk:.1f}% (Target: 20-30%)")
        
        high_risk = (data['ppd_risk_level'].isin(['High Risk', 'Very High Risk'])).mean() * 100
        print(f"High/Very High Risk: {high_risk:.1f}% (Target: 12-20%)\n")
        
        print(f"Mean EPDS Score: {data['epds_equivalent_score'].mean():.1f}")
        print(f"Median EPDS Score: {data['epds_equivalent_score'].median():.1f}\n")
        
        # Wealth distribution
        print("=== SOCIOECONOMIC STATUS ===")
        print("Wealth Index Distribution:")
        print(data['wealth_index'].value_counts(normalize=True).mul(100).round(1))
        print(f"\nMedian Income: NPR {data['monthly_household_income_npr'].median():,.0f}")
        
        print(f"\n{'='*70}")
        
        return data


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Generate 100,000+ records
    generator = NepalMaternalHealthDataGenerator(n_samples=100000, random_seed=42)
    
    # Generate complete dataset
    maternal_health_data = generator.generate_complete_dataset()
    
    # Validate against Nepal statistics
    validated_data = generator.validate_dataset(maternal_health_data)
    
    # Save to CSV
    output_file = 'nepal_maternal_mental_health_synthetic_data_claude_comet.csv'
    validated_data.to_csv(output_file, index=False)
    print(f"\nDataset saved to: {output_file}")
    
    # Display sample records
    print(f"\n{'='*70}")
    print("SAMPLE RECORDS (First 5):")
    print(f"{'='*70}\n")
    print(validated_data.head())
    
    # Display high-risk cases
    print(f"\n{'='*70}")
    print("SAMPLE HIGH-RISK CASES:")
    print(f"{'='*70}\n")
    high_risk_cases = validated_data[validated_data['ppd_risk_level'] == 'Very High Risk'].head(3)
    
    for idx, row in high_risk_cases.iterrows():
        print(f"\nCase ID: {row['record_id']}")
        print(f"  Age: {row['age']}, Parity: {row['parity']}, Education: {row['education_level']}")
        print(f"  Partner Support: {row['partner_support_score']}/10, DV: {row['domestic_violence_exposure']}")
        print(f"  Baby Health: {row['baby_health_status']}, Complications: {row['birth_complications']}")
        print(f"  Wealth: {row['wealth_index']}, Food Security: {row['food_security']}")
        print(f"  EPDS Score: {row['epds_equivalent_score']}, Risk Level: {row['ppd_risk_level']}")
    
    print(f"\n{'='*70}")
    print("GENERATION COMPLETE - Dataset ready for fuzzy ML analysis")
    print(f"{'='*70}\n")


NEPAL MATERNAL MENTAL HEALTH SYNTHETIC DATA GENERATOR
Target sample size: 100,000

Generating demographic features...
Generating socioeconomic features...
Generating obstetric features...
Generating psychosocial features...
Generating healthcare access features...
Calculating PPD risk scores...

DATA GENERATION COMPLETE!

DATASET VALIDATION REPORT

Total Records: 100,000

=== DEMOGRAPHIC DISTRIBUTIONS ===
Mean Age: 25.5 (Target: ~26)
Urban: 39.8% (Target: 40%)
Rural: 60.2% (Target: 60%)

Education Distribution:
education_level
Secondary       45.7
Higher          22.5
Primary         20.9
No education    10.9
Name: proportion, dtype: float64

=== OBSTETRIC STATISTICS ===
Mean Parity: 2.30 (Target: ~2.3)
4+ ANC Visits: 84.6% (Target: 80.2%)
Cesarean Rate: 23.7% (Target: 18.2%)
Birth Complications: 30.4% (Target: ~25%)

=== PSYCHOSOCIAL FACTORS ===
Any Domestic Violence: 29.5% (Target: 27.2%)
Mean Partner Support: 6.2/10
Mean Social Isolation: 5.9/10

=== PPD RISK DISTRIBUTION ===
ppd_r